# STEP 3: Measurement Extraction Pipeline

In [5]:
# ==============================
# STEP 3.1: IMPORT MEASUREMENT LIBRARIES
# ==============================
import math
import pandas as pd
from scipy.spatial.distance import euclidean
from sklearn.linear_model import LinearRegression
import json
import os

print("📏 STEP 3: Measurement Extraction Pipeline")
print("=" * 60)

📏 STEP 3: Measurement Extraction Pipeline


In [2]:
# ==============================
# STEP 3.2: MEASUREMENT CONFIGURATION
# ==============================
MEASUREMENT_CONFIG = {
    # Measurement definitions with keypoint pairs
    'side_measurements': {
        'head_length': {'points': [0, 9], 'type': 'distance'},
        'oblique_body_length': {'points': [2, 4], 'type': 'distance'},
        'girth_length': {'points': [5, 6], 'type': 'distance'},
        'chest_depth': {'points': [1, 7], 'type': 'distance'},
        'withers_height': {'points': [1, 8], 'type': 'distance'},
        'rump_angle': {'points': [3, 4], 'type': 'angle'}  # Requires 3 points for angle
    },
    
    'back_measurements': {
        'rump_width': {'points': [0, 1], 'type': 'distance'},
        'hip_width': {'points': [2, 3], 'type': 'distance'},
        'tail_length': {'points': [4, 5], 'type': 'distance'},
        'left_hip_height': {'points': [2, 6], 'type': 'distance'},
        'right_hip_height': {'points': [3, 7], 'type': 'distance'},
        'thurl_width': {'points': [0, 2, 1, 3], 'type': 'width'}  # Special case
    },
    
    # Biological plausibility ranges (cm) - can be updated based on actual data
    'plausibility_ranges': {
        'head_length': (35, 65),
        'oblique_body_length': (120, 180),
        'girth_length': (160, 240),
        'chest_depth': (60, 100),
        'withers_height': (110, 150),
        'rump_angle': (20, 45),  # degrees
        'rump_width': (40, 80),
        'hip_width': (45, 85),
        'tail_length': (30, 70),
        'left_hip_height': (110, 150),
        'right_hip_height': (110, 150),
        'thurl_width': (35, 75)
    }
}

print("✅ Measurement configuration loaded")

✅ Measurement configuration loaded


In [3]:
# ==============================
# STEP 3.3: GEOMETRIC CALCULATION FUNCTIONS
# ==============================
def calculate_distance(point1, point2):
    """Calculate Euclidean distance between two points"""
    return euclidean(point1, point2)

def calculate_angle(point1, vertex, point2):
    """
    Calculate angle at vertex formed by three points
    Returns angle in degrees
    """
    # Create vectors from vertex
    vec1 = point1 - vertex
    vec2 = point2 - vertex
    
    # Calculate dot product and magnitudes
    dot_product = np.dot(vec1, vec2)
    mag1 = np.linalg.norm(vec1)
    mag2 = np.linalg.norm(vec2)
    
    # Avoid division by zero
    if mag1 == 0 or mag2 == 0:
        return 0.0
    
    # Calculate cosine of angle
    cos_angle = dot_product / (mag1 * mag2)
    
    # Clamp to avoid numerical issues
    cos_angle = np.clip(cos_angle, -1.0, 1.0)
    
    # Convert to degrees
    angle_rad = math.acos(cos_angle)
    angle_deg = math.degrees(angle_rad)
    
    return angle_deg

def calculate_thurl_width(points):
    """
    Calculate thurl width as the distance between two lines
    points: [left_rump, left_hip, right_rump, right_hip]
    """
    if len(points) < 4:
        return 0.0
    
    left_rump, left_hip, right_rump, right_hip = points[:4]
    
    # Calculate midpoints of left and right sides
    left_mid = (left_rump + left_hip) / 2
    right_mid = (right_rump + right_hip) / 2
    
    # Thurl width is distance between these midpoints
    return calculate_distance(left_mid, right_mid)

def extract_all_measurements(keypoints, view_type):
    """
    Extract all measurements for a given view type
    Returns dictionary of measurements in pixels
    """
    measurements = {}
    
    if view_type == 'side':
        measurement_defs = MEASUREMENT_CONFIG['side_measurements']
        required_points = 10
    else:  # back view
        measurement_defs = MEASUREMENT_CONFIG['back_measurements']
        required_points = 8
    
    # Check if we have enough keypoints
    if keypoints is None or len(keypoints) < required_points:
        print(f"⚠️ Insufficient keypoints for {view_type} view: {len(keypoints) if keypoints else 0}/{required_points}")
        return measurements
    
    try:
        for measurement_name, config in measurement_defs.items():
            points_needed = config['points']
            measurement_type = config['type']
            
            if measurement_type == 'distance':
                # Simple distance between two points
                if len(points_needed) == 2:
                    point1 = keypoints[points_needed[0]]
                    point2 = keypoints[points_needed[1]]
                    measurements[measurement_name] = calculate_distance(point1, point2)
                
            elif measurement_type == 'angle':
                # Angle calculation (rump angle needs 3 points)
                if len(points_needed) == 2:
                    # For rump angle, we need points 3, 4 and an additional reference
                    if measurement_name == 'rump_angle':
                        # Use points 3, 4 and create a horizontal reference from point 3
                        point1 = keypoints[3]  # Rump point 1
                        vertex = keypoints[4]   # Rump point 2 (vertex)
                        # Create horizontal reference point
                        point2 = np.array([vertex[0] + 100, vertex[1]])  # Horizontal line
                        measurements[measurement_name] = calculate_angle(point1, vertex, point2)
                
            elif measurement_type == 'width':
                # Thurl width calculation
                if len(points_needed) == 4:
                    points_list = [keypoints[i] for i in points_needed]
                    measurements[measurement_name] = calculate_thurl_width(points_list)
        
        print(f"✅ Extracted {len(measurements)} measurements from {view_type} view")
        
    except Exception as e:
        print(f"❌ Error extracting measurements: {e}")
    
    return measurements

In [6]:
# ==============================
# STEP 3.4: LOAD EXCEL MEASUREMENTS FOR SCALING
# ==============================
def load_excel_measurements():
    """Load ground truth measurements from Excel file"""
    print("\n📊 Loading Excel measurements for scaling...")
    
    try:
        # Try different possible file locations and names
        excel_paths = [
            'data/measurements.xlsx',
            'measurements.xlsx',
            'data/animal_measurements.xlsx'
        ]
        
        excel_path = None
        for path in excel_paths:
            if os.path.exists(path):
                excel_path = path
                break
        
        if not excel_path:
            print("❌ Excel measurements file not found")
            return None
        
        df = pd.read_excel(excel_path)
        df['image_id'] = df['image_id'].astype(str)
        
        print(f"✅ Excel data loaded: {len(df)} records")
        print(f"📋 Columns: {list(df.columns)}")
        
        # Display sample of the data
        print("\n📄 Sample data from Excel:")
        print(df.head(3).to_string(index=False))
        
        return df
        
    except Exception as e:
        print(f"❌ Error loading Excel file: {e}")
        return None

# Load Excel measurements
excel_df = load_excel_measurements()


📊 Loading Excel measurements for scaling...
✅ Excel data loaded: 138 records
📋 Columns: ['image_id', 'oblique_body_length_cm', 'withers_height_cm', 'girth_length_cm']

📄 Sample data from Excel:
image_id  oblique_body_length_cm  withers_height_cm  girth_length_cm
       1                     161                124              190
       2                     152                120              183
       3                     151                124              179


In [7]:
# ==============================
# STEP 3.5: SCALING FACTOR CALCULATION
# ==============================
def calculate_scaling_factors():
    """
    Calculate pixel-to-cm scaling factors using Excel ground truth
    Returns dictionary of scaling factors for each measurement
    """
    print("\n🔧 Calculating scaling factors...")
    
    if excel_df is None:
        print("❌ No Excel data available for scaling calculation")
        return None
    
    scaling_factors = {}
    scaling_data = []
    
    # Process each image in Excel data
    for idx, row in excel_df.iterrows():
        image_id = str(row['image_id'])
        
        # Try to find corresponding keypoints for this image
        keypoints_side = load_keypoints_for_image(image_id, 'side')
        keypoints_back = load_keypoints_for_image(image_id, 'back')
        
        record = {'image_id': image_id}
        
        # Extract pixel measurements
        if keypoints_side is not None:
            pixel_measurements_side = extract_all_measurements(keypoints_side, 'side')
            record.update({f"{k}_pixels": v for k, v in pixel_measurements_side.items()})
        
        if keypoints_back is not None:
            pixel_measurements_back = extract_all_measurements(keypoints_back, 'back')
            record.update({f"{k}_pixels": v for k, v in pixel_measurements_back.items()})
        
        # Add ground truth measurements from Excel
        for measurement in MEASUREMENT_CONFIG['side_measurements'].keys():
            if measurement in row:
                record[f"{measurement}_cm"] = row[measurement]
        
        for measurement in MEASUREMENT_CONFIG['back_measurements'].keys():
            if measurement in row:
                record[f"{measurement}_cm"] = row[measurement]
        
        scaling_data.append(record)
    
    # Create scaling dataframe
    scaling_df = pd.DataFrame(scaling_data)
    valid_records = len([r for r in scaling_data if any('_pixels' in k for k in r.keys())])
    print(f"📈 Processed {valid_records} valid image-measurement pairs")
    
    # Calculate scaling factors for each measurement
    for measurement in list(MEASUREMENT_CONFIG['side_measurements'].keys()) + list(MEASUREMENT_CONFIG['back_measurements'].keys()):
        pixel_col = f"{measurement}_pixels"
        cm_col = f"{measurement}_cm"
        
        if pixel_col in scaling_df.columns and cm_col in scaling_df.columns:
            # Remove rows with missing data
            valid_data = scaling_df[[pixel_col, cm_col]].dropna()
            
            if len(valid_data) > 5:  # Need minimum samples
                X = valid_data[pixel_col].values.reshape(-1, 1)
                y = valid_data[cm_col].values
                
                # Use linear regression for robust scaling
                reg = LinearRegression()
                reg.fit(X, y)
                
                scaling_factors[measurement] = {
                    'slope': reg.coef_[0],
                    'intercept': reg.intercept_,
                    'r_squared': reg.score(X, y),
                    'samples': len(valid_data)
                }
                
                print(f"  {measurement:20}: {reg.coef_[0]:.4f} cm/pixel, R²={reg.score(X, y):.3f} (n={len(valid_data)})")
    
    # Save scaling data for analysis
    scaling_df.to_csv('scaling_calibration_data.csv', index=False)
    print(f"💾 Scaling calibration data saved: scaling_calibration_data.csv")
    
    return scaling_factors

def load_keypoints_for_image(image_id, view_type):
    """Load keypoints for a specific image ID and view type"""
    try:
        if view_type == 'side':
            label_dir = 'data/labels_side'
            model = side_model
        else:
            label_dir = 'data/labels-back'
            model = back_model
        
        # Find label file
        label_path = os.path.join(label_dir, f"{image_id}.txt")
        if not os.path.exists(label_path):
            return None
        
        # Find corresponding image to get dimensions
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']:
            if view_type == 'side':
                test_path = os.path.join('data/side_view', f"{image_id}{ext}")
            else:
                test_path = os.path.join('data/back_view', f"{image_id}{ext}")
            
            if os.path.exists(test_path):
                img_path = test_path
                break
        
        if not img_path:
            return None
        
        # Load image and keypoints using YOLO model
        results = model.predict(img_path, imgsz=640, conf=0.3)
        return extract_keypoints_from_yolo(results)
        
    except Exception as e:
        print(f"❌ Error loading keypoints for {image_id}_{view_type}: {e}")
        return None

# Calculate scaling factors
scaling_factors = calculate_scaling_factors()


🔧 Calculating scaling factors...
❌ Error loading keypoints for 1_side: name 'side_model' is not defined
❌ Error loading keypoints for 1_back: name 'back_model' is not defined
❌ Error loading keypoints for 2_side: name 'side_model' is not defined
❌ Error loading keypoints for 2_back: name 'back_model' is not defined
❌ Error loading keypoints for 3_side: name 'side_model' is not defined
❌ Error loading keypoints for 3_back: name 'back_model' is not defined
❌ Error loading keypoints for 4_side: name 'side_model' is not defined
❌ Error loading keypoints for 4_back: name 'back_model' is not defined
❌ Error loading keypoints for 5_side: name 'side_model' is not defined
❌ Error loading keypoints for 5_back: name 'back_model' is not defined
❌ Error loading keypoints for 6_side: name 'side_model' is not defined
❌ Error loading keypoints for 6_back: name 'back_model' is not defined
❌ Error loading keypoints for 7_side: name 'side_model' is not defined
❌ Error loading keypoints for 7_back: name 